# ChromaBook Advanced Mapping Lab v2
## آزمایشگاه مستقل پیش‌پردازش، مپ رنگ و کیفیت‌سنجی پیکسلی

این Notebook یک نسخه مستقل و یکپارچه است؛ به Notebook قبلی، فایل‌های کد پروژه یا توابع تعریف‌شده در Runtimeهای دیگر وابسته نیست.

### چهار موتور قابل مقایسه
| موتور | روش |
|---|---|
| A — Legacy Baseline | میانگین Lab هر خانه + نزدیک‌ترین رنگ با فاصله ΔE76 |
| B — Cell-wise Perceptual | میانگین خطای پیکسلی ΔE2000 برای هر رنگ پالت |
| C — Edge-Aware | موتور B + هموارسازی فضایی که در مرزهای قوی ضعیف می‌شود |
| D — Region-Aware | هزینه رنگ متوازن‌شده بر اساس نواحی SLIC + مرزهای تصویر و Superpixel |

### پیش‌پردازش قابل تنظیم
- اصلاح روشنایی ناهمگون با تخمین پس‌زمینه روشنایی
- CLAHE روی کانال روشنایی
- Bilateral / Non-local Means / Median denoising
- شارپ‌سازی چندمقیاسی روی روشنایی
- استخراج مرز با Scharr + Canny + بستن شکاف‌های کوچک

### خروجی‌ها و سنجه‌ها
- Pixel Preview و Numbered Grid برای هر چهار موتور
- Edge Map، SLIC Boundary Map و Delta-E Map
- میانگین، میانه و صدک ۹۵ ΔE2000
- SSIM، Edge F1، Singleton Ratio و Transition Rate
- آزمون پایداری در برابر تغییر کوچک روشنایی
- بسته ZIP شامل PNG، CSV، JSON، NPY و فایل‌های باینری سازگار با ساختار شبکه

### شروع
1. Notebook را در Google Colab باز کن و سلول اصلی را اجرا کن.
2. روی **Upload Image** بزن.
3. در صورت نیاز، `palette.json` واقعی را بارگذاری کن.
4. تنظیمات پیش‌پردازش و Mapping را تغییر بده.
5. **Run All Engines** را اجرا و چهار خروجی را مقایسه کن.
6. در پایان **Export Complete ZIP** را بزن.

**مهم:** پالت داخلی ۲۱ رنگ صرفاً نمونه است و کد رسمی Ohuhu محسوب نمی‌شود. برای شماره‌گذاری صحیح، پالت واقعی خودت را بارگذاری کن. شناسه چاپی رنگ از `color.id` یا `id` ورودی حفظ می‌شود.

**محدودیت تفسیری:** SLIC یک over-segmentation کم‌سطح است، نه تشخیص معنایی اشیا. امتیازهای کیفیت ابزار مقایسه‌اند و جای بررسی بصری خروجی چاپی را نمی‌گیرند. موتورهای C و D از ICM برای بهینه‌سازی تقریبی استفاده می‌کنند، نه Graph Cut دقیق.


In [ ]:
# ================================================================
# CHROMABOOK ADVANCED MAPPING LAB v2 — STANDALONE
# One self-contained code cell. No dependency on previous notebooks.
# ================================================================

# 0) Install only missing packages
import importlib.util, subprocess, sys, os, io, json, time, copy, zipfile, hashlib, warnings
from pathlib import Path

_required = {
    "cv2": "opencv-python-headless",
    "skimage": "scikit-image",
    "scipy": "scipy",
    "ipywidgets": "ipywidgets",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "PIL": "Pillow",
}
_missing = [pkg for module, pkg in _required.items() if importlib.util.find_spec(module) is None]
if _missing:
    print("Installing missing packages:", ", ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from PIL import Image, ImageOps, ImageDraw, ImageFont
from scipy import ndimage
from skimage.color import rgb2lab, lab2rgb, deltaE_ciede2000
from skimage.metrics import structural_similarity
from skimage.segmentation import slic, find_boundaries
from IPython.display import display, clear_output

warnings.filterwarnings("ignore", category=RuntimeWarning)
np.random.seed(42)

# ------------------------------------------------
# 1) USER CONFIGURATION
# ------------------------------------------------
IMAGE_PATH = None       # Optional: "/content/my_image.png"
PALETTE_PATH = None     # Optional: "/content/palette.json"
EXPORT_ROOT = Path("/content/chromabook_advanced_exports")
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

# Sample 21-color palette. Replace with your actual Ohuhu / ChromaGrid palette.
DEFAULT_PALETTE = [
    {"id":"01","name":"Black","hex":"#202124"},
    {"id":"02","name":"Dark Gray","hex":"#55565A"},
    {"id":"03","name":"Gray","hex":"#92949A"},
    {"id":"04","name":"Light Gray","hex":"#D5D7DA"},
    {"id":"05","name":"White","hex":"#FFFFFF"},
    {"id":"06","name":"Dark Brown","hex":"#653B2B"},
    {"id":"07","name":"Brown","hex":"#A56A43"},
    {"id":"08","name":"Beige","hex":"#E5C49B"},
    {"id":"09","name":"Dark Red","hex":"#9E2638"},
    {"id":"10","name":"Red","hex":"#E53945"},
    {"id":"11","name":"Orange","hex":"#F47B35"},
    {"id":"12","name":"Yellow","hex":"#F6D64A"},
    {"id":"13","name":"Yellow Green","hex":"#A9C94B"},
    {"id":"14","name":"Green","hex":"#3C9B62"},
    {"id":"15","name":"Dark Green","hex":"#246348"},
    {"id":"16","name":"Teal","hex":"#35AFA4"},
    {"id":"17","name":"Cyan","hex":"#5CC7D6"},
    {"id":"18","name":"Light Blue","hex":"#75AEE8"},
    {"id":"19","name":"Blue","hex":"#3569B7"},
    {"id":"20","name":"Purple","hex":"#8056A6"},
    {"id":"21","name":"Pink","hex":"#E78BAF"},
]

def hex_to_rgb(value):
    s = str(value).strip().replace("#", "")
    if len(s) == 3:
        s = "".join(ch * 2 for ch in s)
    if len(s) != 6:
        raise ValueError(f"HEX نامعتبر: {value}")
    return tuple(int(s[i:i+2], 16) for i in (0, 2, 4))

def _palette_entries(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for key in ("colors", "palette", "items", "data"):
            val = payload.get(key)
            if isinstance(val, list):
                return val
            if isinstance(val, dict):
                for sub in ("colors", "items", "entries"):
                    if isinstance(val.get(sub), list):
                        return val[sub]
        if any(k in payload for k in ("hex", "rgb", "color")):
            return [payload]
    raise ValueError("ساختار palette.json شناخته نشد؛ آرایه یا کلید colors/palette لازم است.")

def normalize_palette(payload):
    entries = _palette_entries(payload)
    out = []
    for i, entry in enumerate(entries):
        if not isinstance(entry, dict):
            raise ValueError(f"رکورد رنگ {i+1} باید object باشد.")
        nested = entry.get("color") if isinstance(entry.get("color"), dict) else {}
        item = {**nested, **entry}
        hx = item.get("hex") or item.get("hexCode") or item.get("hex_code")
        rgbv = item.get("rgb") or item.get("RGB")
        if hx:
            rgb = hex_to_rgb(hx)
        elif isinstance(rgbv, (list, tuple)) and len(rgbv) >= 3:
            rgb = tuple(int(np.clip(v, 0, 255)) for v in rgbv[:3])
            hx = "#{:02X}{:02X}{:02X}".format(*rgb)
        elif isinstance(rgbv, dict):
            rgb = tuple(int(np.clip(rgbv.get(k, 0), 0, 255)) for k in ("r","g","b"))
            hx = "#{:02X}{:02X}{:02X}".format(*rgb)
        elif all(k in item for k in ("r","g","b")):
            rgb = tuple(int(np.clip(item[k], 0, 255)) for k in ("r","g","b"))
            hx = "#{:02X}{:02X}{:02X}".format(*rgb)
        else:
            raise ValueError(f"رنگ {i+1}: فیلد hex یا rgb وجود ندارد.")
        cid = item.get("id", item.get("code", item.get("number")))
        if cid is None or str(cid).strip() == "":
            raise ValueError(
                f"رنگ شماره {i+1}: شناسه چاپی الزامی است؛ "
                "شناسه را از ترتیب آرایه حدس نمی‌زنیم."
            )
        name = item.get("name", item.get("label", f"Color {i+1}"))
        out.append({"id":str(cid), "name":str(name), "hex":str(hx).upper(),
                    "rgb":tuple(int(v) for v in rgb)})
    if len(out) < 2:
        raise ValueError("پالت باید حداقل دو رنگ داشته باشد.")
    if len({x["id"] for x in out}) != len(out):
        raise ValueError("شناسه چاپی رنگ‌ها در پالت تکراری است.")
    if len(out) > 256:
        raise ValueError("برای خروجی uint8 حداکثر ۲۵۶ رنگ پشتیبانی می‌شود.")
    return out

def attach_palette_lab(palette):
    rgb = np.array([p["rgb"] for p in palette], dtype=np.float32) / 255.0
    labs = rgb2lab(rgb.reshape(1, -1, 3)).reshape(-1, 3).astype(np.float32)
    return [{**p, "lab":labs[i]} for i,p in enumerate(palette)]

PALETTE = attach_palette_lab(normalize_palette(DEFAULT_PALETTE))

# ------------------------------------------------
# 2) IMAGE AND PALETTE LOADING
# ------------------------------------------------
pil_original = None
source_filename = None
LAST_RUN = None
ENGINE_RESULTS = {}
LAST_CONFIG = None

def load_image_bytes(data, filename):
    global pil_original, source_filename
    im = Image.open(io.BytesIO(data))
    pil_original = ImageOps.exif_transpose(im).convert("RGBA")
    source_filename = str(filename)
    return pil_original

def upload_image():
    from google.colab import files
    up = files.upload()
    if not up:
        raise RuntimeError("فایلی انتخاب نشد.")
    valid = [(n,b) for n,b in up.items()
             if Path(n).suffix.lower() in {".jpg",".jpeg",".png",".webp",".bmp",".tif",".tiff"}]
    if not valid:
        raise ValueError("فرمت تصویر باید JPG، PNG، WEBP، BMP یا TIFF باشد.")
    return load_image_bytes(valid[0][1], valid[0][0])

def load_image_path(path):
    global pil_original, source_filename
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(path)
    pil_original = ImageOps.exif_transpose(Image.open(p)).convert("RGBA")
    source_filename = p.name
    return pil_original

def upload_palette():
    global PALETTE
    from google.colab import files
    up = files.upload()
    candidates = [(n,b) for n,b in up.items() if n.lower().endswith(".json")]
    if not candidates:
        raise ValueError("یک فایل JSON برای پالت بارگذاری کن.")
    name, data = candidates[0]
    payload = json.loads(data.decode("utf-8-sig"))
    PALETTE = attach_palette_lab(normalize_palette(payload))
    palette_json_box.value = json.dumps(
        [{"id":p["id"],"name":p["name"],"hex":p["hex"]} for p in PALETTE],
        ensure_ascii=False, indent=2
    )
    show_palette()
    return name

if PALETTE_PATH:
    with open(PALETTE_PATH, "r", encoding="utf-8-sig") as f:
        PALETTE = attach_palette_lab(normalize_palette(json.load(f)))
if IMAGE_PATH:
    load_image_path(IMAGE_PATH)

# ------------------------------------------------
# 3) COLOR / IMAGE HELPERS
# ------------------------------------------------
def to_rgb_array(pil_img):
    rgba = pil_img.convert("RGBA")
    bg = Image.new("RGBA", rgba.size, (255,255,255,255))
    return np.asarray(Image.alpha_composite(bg, rgba).convert("RGB"), dtype=np.uint8)

def crop_resize(rgb, p):
    h,w = rgb.shape[:2]
    if p["crop_enabled"]:
        x0 = int(np.clip(p["crop_x0"], 0, 98))
        y0 = int(np.clip(p["crop_y0"], 0, 98))
        x1 = int(np.clip(p["crop_x1"], x0+1, 100))
        y1 = int(np.clip(p["crop_y1"], y0+1, 100))
    else:
        x0,y0,x1,y1 = 0,0,100,100
    xa,ya = round(x0*w/100), round(y0*h/100)
    xb,yb = round(x1*w/100), round(y1*h/100)
    cropped = rgb[ya:max(ya+1,yb), xa:max(xa+1,xb)].copy()
    max_side = int(p["max_side"])
    hh,ww = cropped.shape[:2]
    scale = min(1.0, max_side/max(hh,ww)) if max_side > 0 else 1.0
    if scale < 1:
        cropped = cv2.resize(cropped, (max(16,round(ww*scale)), max(16,round(hh*scale))),
                              interpolation=cv2.INTER_AREA)
    return cropped

def illumination_correction(rgb, strength=0.0, sigma_fraction=0.10):
    if strength <= 0:
        return rgb.copy(), np.zeros(rgb.shape[:2], np.float32)
    lab = rgb2lab(rgb.astype(np.float32)/255.0).astype(np.float32)
    L = lab[:,:,0]
    sigma = max(3.0, min(rgb.shape[:2]) * float(sigma_fraction))
    background = cv2.GaussianBlur(L, (0,0), sigmaX=sigma, sigmaY=sigma)
    target = float(np.median(background))
    corrected = L - float(strength) * (background - target)
    lab[:,:,0] = np.clip(corrected, 0, 100)
    out = np.clip(lab2rgb(lab)*255.0, 0, 255).astype(np.uint8)
    shade = background - target
    lo,hi = np.percentile(shade,[2,98])
    shade_vis = np.clip((shade-lo)/(hi-lo+1e-6)*255,0,255).astype(np.uint8)
    return out, shade_vis

def clahe_luminance(rgb, clip_limit=0.0, tile_grid=8):
    if clip_limit <= 0:
        return rgb.copy()
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=float(clip_limit),
                            tileGridSize=(int(tile_grid),int(tile_grid)))
    lab[:,:,0] = clahe.apply(lab[:,:,0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def denoise_rgb(rgb, method="bilateral", strength=0.0):
    if method == "none" or strength <= 0:
        return rgb.copy()
    if method == "bilateral":
        d = 5 if strength < 35 else 7
        return cv2.bilateralFilter(rgb, d, float(strength), max(1.0,float(strength)/8))
    if method == "nlm":
        h = float(strength)
        return cv2.fastNlMeansDenoisingColored(rgb, None, h, h, 7, 21)
    if method == "median":
        k = 3 if strength < 35 else 5
        return cv2.medianBlur(rgb, k)
    return rgb.copy()

def multiscale_sharpen(rgb, amount_fine=0.0, amount_coarse=0.0):
    if amount_fine <= 0 and amount_coarse <= 0:
        return rgb.copy()
    lab = rgb2lab(rgb.astype(np.float32)/255.0).astype(np.float32)
    L = lab[:,:,0]
    fine = L - cv2.GaussianBlur(L,(0,0),0.8)
    coarse = L - cv2.GaussianBlur(L,(0,0),2.0)
    lab[:,:,0] = np.clip(L + amount_fine*fine + amount_coarse*coarse, 0, 100)
    return np.clip(lab2rgb(lab)*255.0,0,255).astype(np.uint8)

def preprocess_pipeline(pil_img, p):
    raw = crop_resize(to_rgb_array(pil_img), p)
    illum, shade = illumination_correction(raw, p["illumination_strength"], p["illumination_sigma"])
    local = clahe_luminance(illum, p["clahe_clip"], p["clahe_tiles"])
    den = denoise_rgb(local, p["denoise_method"], p["denoise_strength"])
    sharp = multiscale_sharpen(den, p["sharpen_fine"], p["sharpen_coarse"])
    return {"reference":raw, "illumination_corrected":illum, "clahe":local,
            "denoised":den, "processed":sharp, "shade_map":shade}

# ------------------------------------------------
# 4) EDGE DETECTION — independent of color mapping
# ------------------------------------------------
def extract_edges(rgb, low=55, high=145, auto_canny=True, close_iter=1):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    gx = cv2.Scharr(gray, cv2.CV_32F, 1, 0)
    gy = cv2.Scharr(gray, cv2.CV_32F, 0, 1)
    mag = cv2.magnitude(gx,gy)
    scale = max(1e-6, float(np.percentile(mag, 99.5)))
    grad = np.clip(mag/scale,0,1)

    if auto_canny:
        med = float(np.median(gray))
        lo = int(max(0,(1.0-0.33)*med))
        hi = int(min(255,max(lo+1,(1.0+0.33)*med)))
    else:
        lo,hi = int(low),int(high)
        if hi <= lo: hi = min(255,lo+1)

    canny = cv2.Canny(gray,lo,hi,L2gradient=True)
    if close_iter > 0:
        kernel = np.ones((3,3),np.uint8)
        canny = cv2.morphologyEx(canny,cv2.MORPH_CLOSE,kernel,iterations=int(close_iter))
    canny_f = canny.astype(np.float32)/255.0
    combined = np.maximum(grad*0.75,canny_f)
    combined = cv2.GaussianBlur(combined,(0,0),0.65)
    return np.clip(combined,0,1).astype(np.float32), canny, grad

# ------------------------------------------------
# 5) GRID GEOMETRY AND CIEDE2000 UNARY COSTS
# ------------------------------------------------
def grid_geometry(h,w,cols):
    cols = int(np.clip(cols,4,180))
    rows = max(4,int(round(cols*h/w)))
    xe = np.linspace(0,w,cols+1).round().astype(int)
    ye = np.linspace(0,h,rows+1).round().astype(int)
    return rows,cols,xe,ye

def lab_palette():
    return np.stack([p["lab"] for p in PALETTE]).astype(np.float32)

def delta_e_matrix(pixel_labs, palette_labs):
    # skimage broadcasts (N,1,3) against (1,K,3), returning N x K.
    return deltaE_ciede2000(pixel_labs[:,None,:],palette_labs[None,:,:]).astype(np.float32)

def sampled_pixels(arr, max_samples):
    flat = arr.reshape(-1,arr.shape[-1])
    if len(flat) <= max_samples:
        return flat
    idx = np.linspace(0,len(flat)-1,int(max_samples)).astype(np.int32)
    return flat[idx]

def trimmed_column_mean(costs, trim_fraction=0.0):
    frac = float(np.clip(trim_fraction,0,0.40))
    if frac <= 0 or costs.shape[0] < 5:
        return np.mean(costs,axis=0)
    ordered = np.sort(costs,axis=0)
    cut = min(int(len(ordered)*frac),max(0,(len(ordered)-1)//2))
    return np.mean(ordered[cut:len(ordered)-cut],axis=0) if cut else np.mean(costs,axis=0)

def build_cell_unary(processed_lab, cols, max_samples=96, trim_fraction=0.03):
    h,w = processed_lab.shape[:2]
    rows,cols,xe,ye = grid_geometry(h,w,cols)
    pal = lab_palette()
    unary = np.zeros((rows,cols,len(PALETTE)),np.float32)
    for r in range(rows):
        for c in range(cols):
            cell = processed_lab[ye[r]:ye[r+1],xe[c]:xe[c+1]]
            pix = sampled_pixels(cell,max_samples).astype(np.float32)
            costs = delta_e_matrix(pix,pal)
            unary[r,c] = trimmed_column_mean(costs,trim_fraction)
    return unary,(rows,cols,xe,ye)

def build_region_balanced_unary(processed_lab, slic_labels, geometry,
                                max_samples=72, trim_fraction=0.03):
    rows,cols,xe,ye = geometry
    pal = lab_palette()
    out = np.zeros((rows,cols,len(PALETTE)),np.float32)
    for r in range(rows):
        for c in range(cols):
            y0,y1,x0,x1 = ye[r],ye[r+1],xe[c],xe[c+1]
            cell_lab = processed_lab[y0:y1,x0:x1]
            cell_seg = slic_labels[y0:y1,x0:x1]
            ids,counts = np.unique(cell_seg,return_counts=True)
            region_costs,weights = [],[]
            for sid,count in zip(ids,counts):
                region = cell_lab[cell_seg==sid]
                pix = sampled_pixels(region,max(12,min(max_samples,int(max_samples*count/max(1,cell_seg.size)*2))))
                cm = trimmed_column_mean(delta_e_matrix(pix.astype(np.float32),pal),trim_fraction)
                region_costs.append(cm)
                weights.append(np.sqrt(float(count)))
            weights = np.asarray(weights,np.float32)
            weights /= max(1e-8,weights.sum())
            out[r,c] = np.sum(np.asarray(region_costs)*weights[:,None],axis=0)
    return out

def legacy_mean_lab_grid(processed_lab, geometry, return_cost=False):
    rows,cols,xe,ye = geometry
    pal = lab_palette()
    labels = np.zeros((rows,cols),np.int32)
    selected_cost = np.zeros((rows,cols),np.float32)
    for r in range(rows):
        for c in range(cols):
            cell = processed_lab[ye[r]:ye[r+1],xe[c]:xe[c+1]]
            mean_lab = np.mean(cell.reshape(-1,3),axis=0)
            d = np.linalg.norm(pal-mean_lab[None,:],axis=1) # legacy ΔE76
            labels[r,c] = int(np.argmin(d))
            selected_cost[r,c] = float(d[labels[r,c]])
    return (labels,selected_cost) if return_cost else labels

# ------------------------------------------------
# 6) BOUNDARY-AWARE SPATIAL REGULARIZATION (ICM)
# ------------------------------------------------
def boundary_pairwise_weights(boundary_map, geometry, beta=4.0):
    rows,cols,xe,ye = geometry
    h,w = boundary_map.shape
    wh = np.ones((rows,max(0,cols-1)),np.float32)
    wv = np.ones((max(0,rows-1),cols),np.float32)
    radius = 2
    for r in range(rows):
        y0,y1 = ye[r],ye[r+1]
        for c in range(cols-1):
            x = int(xe[c+1])
            xa,xb = max(0,x-radius),min(w,x+radius+1)
            score = float(np.mean(boundary_map[y0:y1,xa:xb])) if y1>y0 and xb>xa else 0.0
            wh[r,c] = np.exp(-float(beta)*np.clip(score,0,1))
    for r in range(rows-1):
        y = int(ye[r+1])
        for c in range(cols):
            x0,x1 = xe[c],xe[c+1]
            ya,yb = max(0,y-radius),min(h,y+radius+1)
            score = float(np.mean(boundary_map[ya:yb,x0:x1])) if yb>ya and x1>x0 else 0.0
            wv[r,c] = np.exp(-float(beta)*np.clip(score,0,1))
    return wh,wv

def icm_optimize(unary, wh, wv, lam=1.8, iterations=5):
    rows,cols,k = unary.shape
    labels = np.argmin(unary,axis=2).astype(np.int32)
    label_ids = np.arange(k)
    for _ in range(int(iterations)):
        changes = 0
        # Raster scan coordinate descent; deterministic and easy to reproduce.
        for r in range(rows):
            for c in range(cols):
                costs = unary[r,c].copy()
                if c > 0:
                    costs += lam*wh[r,c-1]*(label_ids != labels[r,c-1])
                if c < cols-1:
                    costs += lam*wh[r,c]*(label_ids != labels[r,c+1])
                if r > 0:
                    costs += lam*wv[r-1,c]*(label_ids != labels[r-1,c])
                if r < rows-1:
                    costs += lam*wv[r,c]*(label_ids != labels[r+1,c])
                new = int(np.argmin(costs))
                if new != labels[r,c]:
                    labels[r,c] = new
                    changes += 1
        if changes == 0:
            break
    return labels

# ------------------------------------------------
# 7) SLIC SUPERPIXELS
# ------------------------------------------------
def make_slic(processed_rgb, n_segments=320, compactness=10.0):
    labels = slic(
        processed_rgb.astype(np.float32)/255.0,
        n_segments=int(n_segments),
        compactness=float(compactness),
        sigma=0,
        start_label=0,
        convert2lab=True,
        enforce_connectivity=True,
        channel_axis=-1,
    ).astype(np.int32)
    b = find_boundaries(labels,mode="thick").astype(np.uint8)
    b = cv2.dilate(b,np.ones((3,3),np.uint8),iterations=1)
    b = cv2.GaussianBlur(b.astype(np.float32),(0,0),0.8)
    if b.max() > 0: b /= b.max()
    return labels,b

# ------------------------------------------------
# 8) RENDERING AND QUALITY METRICS
# ------------------------------------------------
def render_pixel_preview(labels, geometry):
    rows,cols,xe,ye = geometry
    h,w = int(ye[-1]),int(xe[-1])
    out = np.full((h,w,3),255,np.uint8)
    for r in range(rows):
        for c in range(cols):
            out[ye[r]:ye[r+1],xe[c]:xe[c+1]] = PALETTE[int(labels[r,c])]["rgb"]
    return out

def render_numbered_grid(labels, cell_px=44, line_width=2):
    rows,cols = labels.shape
    cp = int(np.clip(cell_px,20,100))
    canvas = Image.new("RGB",(cols*cp+1,rows*cp+1),"white")
    draw = ImageDraw.Draw(canvas)
    font = None
    for fp in ("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
               "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf"):
        if os.path.exists(fp):
            try:
                font = ImageFont.truetype(fp,max(9,int(cp*0.34)))
                break
            except Exception:
                pass
    if font is None: font = ImageFont.load_default()
    for r in range(rows):
        for c in range(cols):
            x0,y0 = c*cp,r*cp
            x1,y1 = x0+cp,y0+cp
            draw.rectangle([x0,y0,x1,y1],fill="white",outline=(15,15,15),width=int(line_width))
            text = str(PALETTE[int(labels[r,c])]["id"])
            box = draw.textbbox((0,0),text,font=font)
            tw,th = box[2]-box[0],box[3]-box[1]
            draw.text((x0+(cp-tw)/2-box[0],y0+(cp-th)/2-box[1]),
                      text,fill=(10,10,10),font=font)
    return np.asarray(canvas)

def grid_metrics(labels):
    total = int(labels.size)
    singleton = 0
    tiny = 0
    structure = np.array([[0,1,0],[1,1,1],[0,1,0]],dtype=np.uint8)
    for color_idx in np.unique(labels):
        cc,n = ndimage.label(labels==color_idx,structure=structure)
        if n:
            sizes = np.bincount(cc.ravel())[1:]
            singleton += int(np.sum(sizes==1))
            tiny += int(np.sum(sizes<=2))
    htrans = int(np.sum(labels[:,1:] != labels[:,:-1]))
    vtrans = int(np.sum(labels[1:,:] != labels[:-1,:]))
    adj = labels.shape[0]*max(0,labels.shape[1]-1)+max(0,labels.shape[0]-1)*labels.shape[1]
    return {
        "singleton_ratio_pct":100.0*singleton/max(1,total),
        "tiny_component_ratio_pct":100.0*tiny/max(1,total),
        "transition_rate_pct":100.0*(htrans+vtrans)/max(1,adj),
    }

def edge_f1(reference_rgb, mapped_rgb):
    a = cv2.Canny(cv2.cvtColor(reference_rgb,cv2.COLOR_RGB2GRAY),55,150)
    b = cv2.Canny(cv2.cvtColor(mapped_rgb,cv2.COLOR_RGB2GRAY),55,150)
    kernel = np.ones((3,3),np.uint8)
    ad = cv2.dilate(a,kernel,iterations=1)>0
    bd = cv2.dilate(b,kernel,iterations=1)>0
    ab,bb = a>0,b>0
    precision = float(np.sum(bb & ad)/max(1,np.sum(bb)))
    recall = float(np.sum(ab & bd)/max(1,np.sum(ab)))
    f1 = 2*precision*recall/max(1e-12,precision+recall)
    return f1,precision,recall

def evaluate_grid(reference_rgb, processed_rgb, mapped_rgb, labels):
    ref_lab = rgb2lab(reference_rgb.astype(np.float32)/255.0)
    proc_lab = rgb2lab(processed_rgb.astype(np.float32)/255.0)
    map_lab = rgb2lab(mapped_rgb.astype(np.float32)/255.0)
    de_ref = deltaE_ciede2000(ref_lab,map_lab)
    de_proc = deltaE_ciede2000(proc_lab,map_lab)
    try:
        ssim = float(structural_similarity(reference_rgb,mapped_rgb,
                                           channel_axis=-1,data_range=255,win_size=7))
    except Exception:
        ssim = float("nan")
    f1,prec,rec = edge_f1(reference_rgb,mapped_rgb)
    used = len(np.unique(labels))
    m = {
        "DeltaE00_mean_vs_source":float(np.mean(de_ref)),
        "DeltaE00_median_vs_source":float(np.median(de_ref)),
        "DeltaE00_p95_vs_source":float(np.percentile(de_ref,95)),
        "DeltaE00_mean_vs_processed":float(np.mean(de_proc)),
        "SSIM_vs_source":ssim,
        "Edge_F1_1px":f1,
        "Edge_precision":prec,
        "Edge_recall":rec,
        "colors_used":int(used),
        **grid_metrics(labels),
    }
    return m,de_ref

def delta_map_rgb(de):
    vmax = max(1.0,float(np.percentile(de,99)))
    x = np.clip(de/vmax*255,0,255).astype(np.uint8)
    return cv2.cvtColor(cv2.applyColorMap(x,cv2.COLORMAP_INFERNO),cv2.COLOR_BGR2RGB)

def palette_usage(labels):
    u,c = np.unique(labels,return_counts=True)
    return pd.DataFrame([
        {"Color ID":PALETTE[int(i)]["id"],"Name":PALETTE[int(i)]["name"],
         "HEX":PALETTE[int(i)]["hex"],"Cells":int(n),
         "Share %":100.0*float(n)/labels.size}
        for i,n in zip(u,c)
    ]).sort_values("Cells",ascending=False).reset_index(drop=True)

# ------------------------------------------------
# 9) ENGINE RUNNER
# ------------------------------------------------
ENGINE_NAMES = {
    "A":"A — Legacy Mean-Lab / ΔE76",
    "B":"B — Cell-wise CIEDE2000",
    "C":"C — Edge-Aware ICM",
    "D":"D — SLIC Region-Aware ICM",
}

def map_one_engine(key, processed_lab, geometry, unary, edge_map,
                   slic_labels=None, slic_boundary=None, p=None):
    rows,cols,xe,ye = geometry
    if key == "A":
        labels,selected_cost = legacy_mean_lab_grid(processed_lab,geometry,return_cost=True)
    elif key == "B":
        labels = np.argmin(unary,axis=2).astype(np.int32)
        selected_cost = np.take_along_axis(unary,labels[:,:,None],axis=2)[:,:,0]
    elif key == "C":
        wh,wv = boundary_pairwise_weights(edge_map,geometry,p["boundary_beta"])
        labels = icm_optimize(unary,wh,wv,p["smoothness_lambda"],p["icm_iterations"])
        selected_cost = np.take_along_axis(unary,labels[:,:,None],axis=2)[:,:,0]
    elif key == "D":
        if slic_labels is None:
            raise RuntimeError("SLIC labels برای موتور D ساخته نشده‌اند.")
        region_unary = build_region_balanced_unary(
            processed_lab,slic_labels,geometry,p["max_samples"],p["trim_fraction"])
        blend = float(p["region_balance"])
        d_unary = (1.0-blend)*unary + blend*region_unary
        combined = np.maximum(edge_map,0.95*slic_boundary)
        wh,wv = boundary_pairwise_weights(combined,geometry,p["boundary_beta"])
        labels = icm_optimize(d_unary,wh,wv,p["smoothness_lambda"],p["icm_iterations"])
        selected_cost = np.take_along_axis(d_unary,labels[:,:,None],axis=2)[:,:,0]
    else:
        raise ValueError(key)
    mapped = render_pixel_preview(labels,geometry)
    numbered = render_numbered_grid(labels,p["numbered_cell_px"],p["numbered_line_width"])
    metrics,de = evaluate_grid(CURRENT_PIPELINE["reference"],CURRENT_PIPELINE["processed"],mapped,labels)
    return {
        "key":key,"name":ENGINE_NAMES[key],"labels":labels,"pixel":mapped,
        "numbered":numbered,"metrics":metrics,"delta_map":de,
        "selected_cell_cost":selected_cost,
    }

def run_all_engines(p):
    global CURRENT_PIPELINE
    if pil_original is None:
        raise RuntimeError("ابتدا تصویر را بارگذاری کن.")
    t0 = time.time()
    pipe = preprocess_pipeline(pil_original,p)
    CURRENT_PIPELINE = pipe
    processed = pipe["processed"]
    processed_lab = rgb2lab(processed.astype(np.float32)/255.0).astype(np.float32)
    edge_map,canny,gradient = extract_edges(
        processed,p["canny_low"],p["canny_high"],p["auto_canny"],p["edge_close"])
    unary,geometry = build_cell_unary(
        processed_lab,p["grid_cols"],p["max_samples"],p["trim_fraction"])
    slic_labels,slic_boundary = make_slic(processed,p["slic_segments"],p["slic_compactness"])
    results = {}
    for key in ("A","B","C","D"):
        start = time.time()
        results[key] = map_one_engine(key,processed_lab,geometry,unary,edge_map,
                                      slic_labels,slic_boundary,p)
        results[key]["runtime_seconds"] = time.time()-start
        results[key]["metrics"]["runtime_seconds"] = results[key]["runtime_seconds"]
    pipe["edge_map"] = edge_map
    pipe["canny"] = canny
    pipe["gradient"] = gradient
    pipe["slic_labels"] = slic_labels
    pipe["slic_boundary"] = slic_boundary
    pipe["geometry"] = geometry
    pipe["unary"] = unary
    pipe["processed_lab"] = processed_lab
    return {"pipeline":pipe,"engines":results,"params":dict(p),
            "total_runtime_seconds":time.time()-t0}

# ------------------------------------------------
# 10) ROBUSTNESS / STABILITY TEST
# ------------------------------------------------
def run_stability_test(engine_key, run_data):
    pipe = run_data["pipeline"]
    p = run_data["params"]
    base = run_data["engines"][engine_key]["labels"]
    lab0 = pipe["processed_lab"]
    geometry = pipe["geometry"]
    edge_map = pipe["edge_map"]
    slic_labels = pipe["slic_labels"]
    slic_boundary = pipe["slic_boundary"]
    agreements = []
    for delta_L in (-1.0,1.0):
        pert = lab0.copy()
        pert[:,:,0] = np.clip(pert[:,:,0]+delta_L,0,100)
        unary,_ = build_cell_unary(pert,p["grid_cols"],p["max_samples"],p["trim_fraction"])
        out = map_one_engine(engine_key,pert,geometry,unary,edge_map,
                             slic_labels,slic_boundary,p)
        agreements.append(float(np.mean(out["labels"]==base))*100.0)
    return {
        "Engine":ENGINE_NAMES[engine_key],
        "Agreement after L* -1":agreements[0],
        "Agreement after L* +1":agreements[1],
        "Mean stability %":float(np.mean(agreements)),
    }

# ------------------------------------------------
# 11) VISUALIZATION
# ------------------------------------------------
def show_palette():
    sw = np.zeros((48,len(PALETTE)*42,3),np.uint8)
    for i,x in enumerate(PALETTE):
        sw[:,i*42:(i+1)*42] = x["rgb"]
    plt.figure(figsize=(max(10,len(PALETTE)*0.45),1.2))
    plt.imshow(sw); plt.axis("off"); plt.title(f"Active palette — {len(PALETTE)} colors")
    plt.show()
    display(pd.DataFrame([{"ID":x["id"],"Name":x["name"],"HEX":x["hex"]} for x in PALETTE]))

def show_preprocessing(pipe):
    fig,ax = plt.subplots(2,3,figsize=(17,9))
    panels = [
        (pipe["reference"],"Cropped / resized source",None),
        (pipe["illumination_corrected"],"Illumination corrected",None),
        (pipe["clahe"],"CLAHE",None),
        (pipe["processed"],"Final preprocessed",None),
        (pipe["edge_map"],"Combined edge strength","gray"),
        (pipe["slic_boundary"],"SLIC boundaries","gray"),
    ]
    for a,(im,title,cmap) in zip(ax.flat,panels):
        a.imshow(im,cmap=cmap); a.set_title(title); a.axis("off")
    plt.tight_layout(); plt.show()

def show_all_engine_montage(run_data):
    engines = run_data["engines"]
    fig,ax = plt.subplots(2,4,figsize=(21,10))
    for col,key in enumerate(("A","B","C","D")):
        ax[0,col].imshow(engines[key]["pixel"])
        ax[0,col].set_title(ENGINE_NAMES[key]+"\nPixel Preview")
        ax[0,col].axis("off")
        ax[1,col].imshow(engines[key]["numbered"])
        ax[1,col].set_title("Numbered Grid")
        ax[1,col].axis("off")
    plt.tight_layout(); plt.show()

def show_engine_detail(key,run_data):
    pipe = run_data["pipeline"]
    result = run_data["engines"][key]
    fig,ax = plt.subplots(2,3,figsize=(17,10))
    panels = [
        (pipe["reference"],"Source",None),
        (pipe["processed"],"Preprocessed",None),
        (result["pixel"],"Pixel Preview",None),
        (result["numbered"],"Numbered Grid",None),
        (delta_map_rgb(result["delta_map"]),"Delta-E 2000 Map",None),
        (pipe["edge_map"],"Edge Map","gray"),
    ]
    for a,(im,title,cmap) in zip(ax.flat,panels):
        a.imshow(im,cmap=cmap); a.set_title(title); a.axis("off")
    plt.tight_layout(); plt.show()
    print(ENGINE_NAMES[key])
    display(pd.DataFrame([{"Metric":k,"Value":v} for k,v in result["metrics"].items()]))
    print("Palette usage:")
    display(palette_usage(result["labels"]))

# ------------------------------------------------
# 12) EXPORT PACKAGE
# ------------------------------------------------
def export_complete_package(run_data):
    stamp = time.strftime("%Y%m%d_%H%M%S")
    stem = Path(source_filename or "image").stem[:50] or "image"
    root = EXPORT_ROOT / f"{stem}_{stamp}"
    root.mkdir(parents=True,exist_ok=True)
    pipe = run_data["pipeline"]

    def save_img(path,arr):
        Image.fromarray(np.asarray(arr).astype(np.uint8)).save(path)

    save_img(root/"00_source_cropped.png",pipe["reference"])
    save_img(root/"01_illumination_corrected.png",pipe["illumination_corrected"])
    save_img(root/"02_clahe.png",pipe["clahe"])
    save_img(root/"03_preprocessed.png",pipe["processed"])
    save_img(root/"04_edge_strength.png",(pipe["edge_map"]*255).astype(np.uint8))
    save_img(root/"05_canny.png",pipe["canny"])
    save_img(root/"06_slic_boundaries.png",(pipe["slic_boundary"]*255).astype(np.uint8))

    all_metrics = []
    for key,res in run_data["engines"].items():
        folder = root/f"Engine_{key}"
        folder.mkdir(exist_ok=True)
        save_img(folder/"pixel_preview.png",res["pixel"])
        save_img(folder/"numbered_grid.png",res["numbered"])
        save_img(folder/"deltaE00_map.png",delta_map_rgb(res["delta_map"]))
        np.save(folder/"grid_indices.npy",res["labels"].astype(np.uint8))
        res["labels"].astype(np.uint8).tofile(folder/"grid_indices.bin")
        pd.DataFrame(res["labels"]).to_csv(folder/"grid_palette_indices.csv",index=False,header=False)
        ids = np.empty(res["labels"].shape,dtype=object)
        for rr in range(res["labels"].shape[0]):
            for cc in range(res["labels"].shape[1]):
                ids[rr,cc] = PALETTE[int(res["labels"][rr,cc])]["id"]
        pd.DataFrame(ids).to_csv(folder/"grid_color_ids.csv",index=False,header=False,encoding="utf-8-sig")
        pd.DataFrame(palette_usage(res["labels"])).to_csv(folder/"palette_usage.csv",index=False,encoding="utf-8-sig")
        # Per-cell selected unary cost, stored as Float32 Little Endian.
        res["selected_cell_cost"].astype("<f4").tofile(folder/"cell_delta_e.bin")
        meta = {
            "engine":key,"engine_name":res["name"],
            "rows":int(res["labels"].shape[0]),"cols":int(res["labels"].shape[1]),
            "storage":"row-major uint8 palette indices",
            "cell_delta_e":"float32 little-endian selected-label unary cost",
            "palette_ids":[p["id"] for p in PALETTE],
        }
        (folder/"metadata.json").write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding="utf-8")
        (folder/"metrics.json").write_text(json.dumps(res["metrics"],ensure_ascii=False,indent=2),encoding="utf-8")
        all_metrics.append({"Engine":key,"Name":res["name"],**res["metrics"]})

    pd.DataFrame(all_metrics).to_csv(root/"engine_comparison_metrics.csv",index=False,encoding="utf-8-sig")
    palette_json = [{"id":p["id"],"name":p["name"],"hex":p["hex"],"rgb":list(p["rgb"])} for p in PALETTE]
    (root/"palette.json").write_text(json.dumps({"colors":palette_json},ensure_ascii=False,indent=2),encoding="utf-8")
    (root/"config.json").write_text(json.dumps(run_data["params"],ensure_ascii=False,indent=2),encoding="utf-8")
    readme = f"""ChromaBook Advanced Mapping Lab v2 export
Source: {source_filename}
Created: {stamp}
Engines: A legacy mean-Lab; B cell-wise CIEDE2000; C edge-aware ICM; D SLIC region-aware ICM.
Grid: {run_data['engines']['A']['labels'].shape[1]} columns x {run_data['engines']['A']['labels'].shape[0]} rows.
grid_indices.bin: uint8 row-major palette indices (not color IDs).
cell_delta_e.bin: float32 little-endian selected-label unary costs.
The sample palette is illustrative unless replaced by the user's actual palette.json.
"""
    (root/"README.txt").write_text(readme,encoding="utf-8")
    zip_path = EXPORT_ROOT/f"{stem}_{stamp}_ChromaBook_Advanced_Mapping.zip"
    with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
        for fp in root.rglob("*"):
            if fp.is_file():
                z.write(fp,fp.relative_to(root))
    print("ZIP ready:",zip_path)
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception:
        print("فایل در این مسیر ذخیره شد:",zip_path)
    return zip_path

def export_config_json(run_data):
    stamp = time.strftime("%Y%m%d_%H%M%S")
    path = EXPORT_ROOT/f"chromabook_mapping_config_{stamp}.json"
    payload = {
        "schema":"chromabook-advanced-mapping-v2",
        "created_at":stamp,
        "source_filename":source_filename,
        "params":run_data["params"],
        "palette":[{"id":p["id"],"name":p["name"],"hex":p["hex"]} for p in PALETTE],
    }
    path.write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding="utf-8")
    try:
        from google.colab import files
        files.download(str(path))
    except Exception:
        print(path)

# ------------------------------------------------
# 13) UI WIDGETS
# ------------------------------------------------
style = {"description_width":"155px"}
wide = widgets.Layout(width="350px")

upload_btn = widgets.Button(description="Upload Image",button_style="success",icon="upload")
palette_btn = widgets.Button(description="Upload Palette JSON",icon="palette")
apply_palette_btn = widgets.Button(description="Apply Edited Palette JSON",button_style="info",icon="check")
run_btn = widgets.Button(description="Run All Engines",button_style="success",icon="play")
inspect_btn = widgets.Button(description="Inspect Selected Engine",icon="search")
stability_btn = widgets.Button(description="Stability Test ±1 L*",button_style="warning",icon="refresh")
export_btn = widgets.Button(description="Export Complete ZIP",button_style="primary",icon="download")
config_btn = widgets.Button(description="Export Config JSON",icon="file")

# Preprocessing controls
max_side_w = widgets.IntSlider(value=900,min=256,max=1800,step=32,description="Max analysis side",style=style,layout=wide,continuous_update=False)
crop_w = widgets.Checkbox(value=False,description="Enable crop")
crop_x0_w = widgets.IntSlider(value=0,min=0,max=98,description="Crop left %",style=style,layout=wide,continuous_update=False)
crop_y0_w = widgets.IntSlider(value=0,min=0,max=98,description="Crop top %",style=style,layout=wide,continuous_update=False)
crop_x1_w = widgets.IntSlider(value=100,min=2,max=100,description="Crop right %",style=style,layout=wide,continuous_update=False)
crop_y1_w = widgets.IntSlider(value=100,min=2,max=100,description="Crop bottom %",style=style,layout=wide,continuous_update=False)
illum_w = widgets.FloatSlider(value=0.25,min=0,max=1,step=0.025,description="Illumination correction",style=style,layout=wide,continuous_update=False)
illum_sigma_w = widgets.FloatSlider(value=0.10,min=0.02,max=0.25,step=0.01,description="Illumination scale",style=style,layout=wide,continuous_update=False)
clahe_w = widgets.FloatSlider(value=1.5,min=0,max=5,step=0.1,description="CLAHE clip limit",style=style,layout=wide,continuous_update=False)
clahe_tiles_w = widgets.IntSlider(value=8,min=4,max=16,step=1,description="CLAHE tile grid",style=style,layout=wide,continuous_update=False)
denoise_method_w = widgets.Dropdown(options=[("None","none"),("Bilateral","bilateral"),("Non-local Means","nlm"),("Median","median")],value="bilateral",description="Denoise method",style=style,layout=wide)
denoise_strength_w = widgets.FloatSlider(value=18,min=0,max=60,step=1,description="Denoise strength",style=style,layout=wide,continuous_update=False)
sharpen_fine_w = widgets.FloatSlider(value=0.15,min=0,max=1.5,step=0.025,description="Fine sharpen",style=style,layout=wide,continuous_update=False)
sharpen_coarse_w = widgets.FloatSlider(value=0.05,min=0,max=1.0,step=0.025,description="Coarse sharpen",style=style,layout=wide,continuous_update=False)

# Edge controls
auto_canny_w = widgets.Checkbox(value=True,description="Auto Canny thresholds")
canny_low_w = widgets.IntSlider(value=55,min=0,max=254,description="Canny low",style=style,layout=wide,continuous_update=False)
canny_high_w = widgets.IntSlider(value=145,min=1,max=255,description="Canny high",style=style,layout=wide,continuous_update=False)
edge_close_w = widgets.IntSlider(value=1,min=0,max=3,description="Close iterations",style=style,layout=wide,continuous_update=False)

# Mapping controls
grid_cols_w = widgets.IntSlider(value=40,min=8,max=120,step=1,description="Grid columns",style=style,layout=wide,continuous_update=False)
max_samples_w = widgets.IntSlider(value=96,min=16,max=256,step=8,description="Samples / cell",style=style,layout=wide,continuous_update=False)
trim_w = widgets.FloatSlider(value=0.03,min=0,max=0.30,step=0.01,description="Trim outliers",style=style,layout=wide,continuous_update=False)
smooth_lambda_w = widgets.FloatSlider(value=1.8,min=0,max=8,step=0.1,description="Smoothness λ",style=style,layout=wide,continuous_update=False)
beta_w = widgets.FloatSlider(value=4.0,min=0,max=10,step=0.25,description="Boundary sensitivity β",style=style,layout=wide,continuous_update=False)
icm_iter_w = widgets.IntSlider(value=5,min=1,max=12,step=1,description="ICM iterations",style=style,layout=wide,continuous_update=False)
slic_segments_w = widgets.IntSlider(value=320,min=80,max=900,step=20,description="SLIC segments",style=style,layout=wide,continuous_update=False)
slic_compact_w = widgets.FloatSlider(value=10,min=1,max=30,step=0.5,description="SLIC compactness",style=style,layout=wide,continuous_update=False)
region_balance_w = widgets.FloatSlider(value=0.25,min=0,max=0.8,step=0.025,description="Region balance",style=style,layout=wide,continuous_update=False)
number_cell_w = widgets.IntSlider(value=44,min=20,max=90,step=2,description="Number cell px",style=style,layout=wide,continuous_update=False)
number_line_w = widgets.IntSlider(value=2,min=1,max=5,description="Grid line width",style=style,layout=wide,continuous_update=False)

engine_select_w = widgets.Dropdown(options=[(v,k) for k,v in ENGINE_NAMES.items()],value="D",description="Inspect engine",style=style,layout=wide)
palette_json_box = widgets.Textarea(value=json.dumps(DEFAULT_PALETTE,ensure_ascii=False,indent=2),
                                    description="Palette JSON",layout=widgets.Layout(width="100%",height="150px"),
                                    style={"description_width":"100px"})

status_out = widgets.Output()
preprocess_out = widgets.Output()
montage_out = widgets.Output()
detail_out = widgets.Output()
metrics_out = widgets.Output()
stability_out = widgets.Output()

def get_params():
    return {
        "max_side":int(max_side_w.value),
        "crop_enabled":bool(crop_w.value),
        "crop_x0":int(crop_x0_w.value),"crop_y0":int(crop_y0_w.value),
        "crop_x1":int(crop_x1_w.value),"crop_y1":int(crop_y1_w.value),
        "illumination_strength":float(illum_w.value),
        "illumination_sigma":float(illum_sigma_w.value),
        "clahe_clip":float(clahe_w.value),"clahe_tiles":int(clahe_tiles_w.value),
        "denoise_method":str(denoise_method_w.value),
        "denoise_strength":float(denoise_strength_w.value),
        "sharpen_fine":float(sharpen_fine_w.value),
        "sharpen_coarse":float(sharpen_coarse_w.value),
        "auto_canny":bool(auto_canny_w.value),
        "canny_low":int(canny_low_w.value),"canny_high":int(canny_high_w.value),
        "edge_close":int(edge_close_w.value),
        "grid_cols":int(grid_cols_w.value),
        "max_samples":int(max_samples_w.value),
        "trim_fraction":float(trim_w.value),
        "smoothness_lambda":float(smooth_lambda_w.value),
        "boundary_beta":float(beta_w.value),
        "icm_iterations":int(icm_iter_w.value),
        "slic_segments":int(slic_segments_w.value),
        "slic_compactness":float(slic_compact_w.value),
        "region_balance":float(region_balance_w.value),
        "numbered_cell_px":int(number_cell_w.value),
        "numbered_line_width":int(number_line_w.value),
    }

def on_upload(_):
    try:
        upload_image()
        with status_out:
            clear_output(wait=True)
            print(f"تصویر دریافت شد: {source_filename} | {pil_original.width} × {pil_original.height}")
    except Exception as e:
        with status_out:
            clear_output(wait=True); print("Upload error:",repr(e))

def on_palette(_):
    try:
        name = upload_palette()
        with status_out:
            clear_output(wait=True); print(f"پالت بارگذاری شد: {name} | {len(PALETTE)} رنگ")
    except Exception as e:
        with status_out:
            clear_output(wait=True); print("Palette error:",repr(e))

def on_apply_palette(_):
    global PALETTE
    try:
        payload = json.loads(palette_json_box.value)
        PALETTE = attach_palette_lab(normalize_palette(payload))
        with status_out:
            clear_output(wait=True)
            print(f"پالت ویرایش‌شده اعمال شد: {len(PALETTE)} رنگ. شناسه‌ها از خود JSON خوانده شدند.")
        with palette_display_out:
            clear_output(wait=True)
            show_palette()
    except Exception as e:
        with status_out:
            clear_output(wait=True); print("Palette JSON error:",repr(e))

def on_run(_):
    global LAST_RUN,ENGINE_RESULTS,LAST_CONFIG
    try:
        if pil_original is None:
            raise RuntimeError("ابتدا تصویر را بارگذاری کن.")
        p = get_params()
        with status_out:
            clear_output(wait=True); print("در حال اجرای چهار موتور؛ بسته به ابعاد تصویر ممکن است کمی زمان ببرد...")
        run_data = run_all_engines(p)
        LAST_RUN = run_data
        ENGINE_RESULTS = run_data["engines"]
        LAST_CONFIG = p
        with status_out:
            clear_output(wait=True)
            print(f"انجام شد | زمان کل: {run_data['total_runtime_seconds']:.1f} ثانیه | "
                  f"شبکه: {run_data['engines']['A']['labels'].shape[1]} ستون × "
                  f"{run_data['engines']['A']['labels'].shape[0]} ردیف")
        with preprocess_out:
            clear_output(wait=True); show_preprocessing(run_data["pipeline"])
        with montage_out:
            clear_output(wait=True); show_all_engine_montage(run_data)
        rows = [{"Engine":k,"Method":ENGINE_NAMES[k],**v["metrics"]} for k,v in run_data["engines"].items()]
        with metrics_out:
            clear_output(wait=True); display(pd.DataFrame(rows).set_index("Engine"))
    except Exception as e:
        with status_out:
            clear_output(wait=True); print("Processing error:",repr(e))
        raise

def on_inspect(_):
    with detail_out:
        clear_output(wait=True)
        if LAST_RUN is None:
            print("ابتدا Run All Engines را اجرا کن.")
        else:
            show_engine_detail(engine_select_w.value,LAST_RUN)

def on_stability(_):
    with stability_out:
        clear_output(wait=True)
        if LAST_RUN is None:
            print("ابتدا Run All Engines را اجرا کن.")
            return
        print("در حال آزمون پایداری در برابر تغییر ±1 واحد L* ...")
        rows = [run_stability_test(k,LAST_RUN) for k in ("B","C","D")]
        display(pd.DataFrame(rows).set_index("Engine"))

def on_export(_):
    if LAST_RUN is None:
        print("ابتدا Run All Engines را اجرا کن.")
        return
    export_complete_package(LAST_RUN)

def on_config(_):
    if LAST_RUN is None:
        print("ابتدا Run All Engines را اجرا کن.")
        return
    export_config_json(LAST_RUN)

upload_btn.on_click(on_upload)
palette_btn.on_click(on_palette)
apply_palette_btn.on_click(on_apply_palette)
run_btn.on_click(on_run)
inspect_btn.on_click(on_inspect)
stability_btn.on_click(on_stability)
export_btn.on_click(on_export)
config_btn.on_click(on_config)

pre_tab = widgets.VBox([
    widgets.HTML("<h3>Preprocessing</h3>"),
    max_side_w,crop_w,crop_x0_w,crop_y0_w,crop_x1_w,crop_y1_w,
    illum_w,illum_sigma_w,clahe_w,clahe_tiles_w,
    denoise_method_w,denoise_strength_w,sharpen_fine_w,sharpen_coarse_w
])
edge_tab = widgets.VBox([
    widgets.HTML("<h3>Edge extraction</h3>"),
    auto_canny_w,canny_low_w,canny_high_w,edge_close_w
])
map_tab = widgets.VBox([
    widgets.HTML("<h3>Grid and mapping</h3>"),
    grid_cols_w,max_samples_w,trim_w,smooth_lambda_w,beta_w,icm_iter_w,
    slic_segments_w,slic_compact_w,region_balance_w,number_cell_w,number_line_w
])
tabs = widgets.Tab(children=[pre_tab,edge_tab,map_tab])
tabs.set_title(0,"Preprocess")
tabs.set_title(1,"Edges")
tabs.set_title(2,"Mapping")

display(widgets.HTML(
    "<h2>ChromaBook Advanced Mapping Lab v2</h2>"
    "<p>چهار موتور مستقل را روی یک تصویر و یک پالت اجرا و مقایسه کن. "
    "ابتدا تصویر را بارگذاری کن؛ سپس تنظیمات را تغییر بده و Run All Engines را بزن.</p>"
))
display(widgets.HBox([upload_btn,palette_btn,apply_palette_btn]))
display(status_out)
display(widgets.HTML("<h3>Active palette — JSON is editable</h3><p>پس از ویرایش دستی JSON، دکمه Apply Edited Palette JSON را بزن. هر رنگ باید شناسه چاپی صریح داشته باشد.</p>"))
display(palette_json_box)
palette_display_out = widgets.Output()
display(palette_display_out)
with palette_display_out:
    show_palette()
display(tabs)
display(widgets.HBox([run_btn,engine_select_w,inspect_btn,stability_btn,export_btn,config_btn]))
display(widgets.HTML("<hr><h3>Preprocessing diagnostics</h3>"))
display(preprocess_out)
display(widgets.HTML("<hr><h3>Four-engine visual comparison</h3>"))
display(montage_out)
display(widgets.HTML("<hr><h3>Engine metrics comparison</h3>"))
display(metrics_out)
display(widgets.HTML("<hr><h3>Selected engine details</h3>"))
display(detail_out)
display(widgets.HTML("<hr><h3>Grid stability test</h3>"))
display(stability_out)

if pil_original is not None:
    with status_out:
        clear_output(wait=True)
        print(f"Image ready: {source_filename} | {pil_original.width} × {pil_original.height}")
else:
    with status_out:
        clear_output(wait=True)
        print("برای شروع روی Upload Image کلیک کن.")

# ------------------------------------------------
# 14) OPTIONAL SELF-TEST — algorithm sanity checks
# ------------------------------------------------
def run_self_test():
    # Small synthetic image and palette mapping sanity test.
    test_rgb = np.zeros((120,160,3),np.uint8)
    test_rgb[:,:80] = (220,40,40)
    test_rgb[:,80:] = (40,80,220)
    test_lab = rgb2lab(test_rgb.astype(np.float32)/255.0)
    test_unary,geom = build_cell_unary(test_lab,16,max_samples=32,trim_fraction=0)
    assert test_unary.shape[0] > 0 and test_unary.shape[1] == 16
    edge,_,_ = extract_edges(test_rgb)
    slic_l,slic_b = make_slic(test_rgb,n_segments=30,compactness=8)
    wh,wv = boundary_pairwise_weights(edge,geom,beta=4)
    labels = icm_optimize(test_unary,wh,wv,lam=1.5,iterations=2)
    assert labels.shape == test_unary.shape[:2]
    assert labels.min() >= 0 and labels.max() < len(PALETTE)
    print("SELF-TEST PASSED — grid, CIEDE2000 costs, SLIC and ICM are operational.")

print("Notebook initialized. Optional: run_self_test()")
